<a href="https://colab.research.google.com/github/jzazueta/data-science-business-econ/blob/main/notebooks/ch09_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 9 — Classification: From Estimate to Decision

**Data Science for Business and Economics: A Python Approach**---Jorge Zazueta Gutiérrez

This notebook mirrors every code example in Chapter 9. Priya Raghunathan, a credit and collections analyst at Halvorsen Industrial Supply, needs a rule that decides which open invoices the collections team should work.

Run the cells in order — later steps depend on objects created earlier.

The chapter's marquee result is that a model scoring 93.63% accuracy is worse than useless on this problem, and that the fix is a threshold rather than a better model.

### Setup — getting the data

This cell is notebook scaffolding and does not appear in the book. It finds the receivables extract in the repository, or downloads it if you are running in Colab. Every later cell reads `halvorsen_receivables.csv` from the working directory.

In [1]:
import os
import urllib.request

DATA = "halvorsen_receivables.csv"
RAW = ("https://raw.githubusercontent.com/jzazueta/data-science-business-econ/"
       "main/data/halvorsen_receivables.csv")

if not os.path.exists(DATA):
    for candidate in ("data/halvorsen_receivables.csv", "../data/halvorsen_receivables.csv"):
        if os.path.exists(candidate):
            DATA = candidate
            break
    else:
        urllib.request.urlretrieve(RAW, DATA)

print("using:", DATA)

using: halvorsen_receivables.csv


## 9.1 From Explaining to Predicting

Priya Raghunathan works in credit and collections at Halvorsen Industrial Supply. Her problem is which open invoices to chase while there is still time to act on them.

### Code Example 9.1 — Loading the receivables extract and counting the outcome

In [2]:
import numpy as np
import pandas as pd

inv = pd.read_csv(DATA)

print(inv.shape)
print(inv["delinquent_90"].value_counts())
print(round(inv["delinquent_90"].mean(), 4))

(12000, 14)
delinquent_90
0    11243
1      757
Name: count, dtype: int64
0.0631


### Code Example 9.2 — Delinquency rate by customer segment

In [3]:
print(inv.groupby("customer_segment")["delinquent_90"]
        .agg(["size", "mean"]).round(4))

                  size    mean
customer_segment              
Independent       3130  0.0741
National Account  4311  0.0538
Regional          4559  0.0643


## 9.2 Logistic Regression and the Odds Ratio

A model of the log odds, fitted with `statsmodels` so the coefficient table and the odds ratios can be read the way Chapter 8 read a regression.

### Code Example 9.3 — Fitting the logistic model in statsmodels

In [4]:
import statsmodels.api as sm

pd.set_option("display.width", 100)
inv["log_amount"] = np.log(inv["invoice_amount"])

cols = ["log_amount", "payment_terms_days", "relationship_months",
        "prior_late_rate", "disputed_line_items"]
X = pd.get_dummies(inv[cols + ["customer_segment", "order_channel"]],
                   drop_first=True).astype(float)
X = sm.add_constant(X)

logit = sm.Logit(inv["delinquent_90"], X).fit(disp=0)
print(logit.summary2().tables[1].round(4))

                                    Coef.  Std.Err.        z   P>|z|  [0.025  0.975]
const                             -8.6543    0.4271 -20.2606  0.0000 -9.4915 -7.8171
log_amount                         0.5125    0.0469  10.9164  0.0000  0.4205  0.6045
payment_terms_days                 0.0137    0.0038   3.6367  0.0003  0.0063  0.0210
relationship_months               -0.0064    0.0013  -4.9461  0.0000 -0.0089 -0.0039
prior_late_rate                    5.3070    0.3024  17.5492  0.0000  4.7143  5.8997
disputed_line_items                1.4391    0.1231  11.6952  0.0000  1.1979  1.6803
customer_segment_National Account -0.6110    0.1437  -4.2508  0.0000 -0.8927 -0.3293
customer_segment_Regional         -0.2245    0.1042  -2.1551  0.0312 -0.4287 -0.0203
order_channel_Phone                1.1028    0.1305   8.4504  0.0000  0.8471  1.3586
order_channel_Sales Rep            0.7024    0.1252   5.6106  0.0000  0.4570  0.9478
order_channel_Web Portal           0.5761    0.1305   4.4141  0.0

### Code Example 9.4 — Converting coefficients into odds ratios

In [5]:
odds = pd.DataFrame({
    "coef": logit.params,
    "odds_ratio": np.exp(logit.params),
}).round(4)
print(odds)
print(round(logit.prsquared, 4))

                                     coef  odds_ratio
const                             -8.6543      0.0002
log_amount                         0.5125      1.6694
payment_terms_days                 0.0137      1.0138
relationship_months               -0.0064      0.9936
prior_late_rate                    5.3070    201.7369
disputed_line_items                1.4391      4.2170
customer_segment_National Account -0.6110      0.5428
customer_segment_Regional         -0.2245      0.7989
order_channel_Phone                1.1028      3.0127
order_channel_Sales Rep            0.7024      2.0186
order_channel_Web Portal           0.5761      1.7792
0.115


## 9.3 Why Accuracy Fails

The split comes before the fitting. Everything from here uses `scikit-learn`.

### Code Example 9.5 — Splitting the extract into training and test sets

In [6]:
from sklearn.model_selection import train_test_split

NUM = ["log_amount", "payment_terms_days", "relationship_months",
       "prior_invoices", "prior_late_rate", "disputed_line_items"]
CAT = ["customer_segment", "region", "order_channel"]

X = inv[NUM + CAT]
y = inv["delinquent_90"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

print("train:", X_train.shape, " test:", X_test.shape)
print("train rate:", round(y_train.mean(), 4),
      " test rate:", round(y_test.mean(), 4))

train: (9000, 9)  test: (3000, 9)
train rate: 0.0631  test rate: 0.063


### Code Example 9.6 — A pipeline that scales, encodes, and fits in one object

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score

prep = ColumnTransformer([
    ("num", StandardScaler(), NUM),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), CAT),
])

clf = Pipeline([
    ("prep", prep),
    ("model", LogisticRegression(max_iter=2000)),
])

clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print("accuracy:", round(accuracy_score(y_test, pred), 4))

accuracy: 0.9363


### Code Example 9.7 — Counting what the classifier flagged

In [8]:
from sklearn.metrics import confusion_matrix

print("invoices flagged:", int(pred.sum()), "of", len(pred))
print("delinquent invoices in test set:", int(y_test.sum()))
print(confusion_matrix(y_test, pred))

invoices flagged: 6 of 3000
delinquent invoices in test set: 189
[[2807    4]
 [ 187    2]]


### Code Example 9.8 — The accuracy of predicting that nobody goes delinquent

In [9]:
naive = np.zeros(len(y_test), dtype=int)
print("accuracy of the do-nothing rule:",
      round(accuracy_score(y_test, naive), 4))

accuracy of the do-nothing rule: 0.937


### Code Example 9.9 — The probabilities behind the labels

In [10]:
proba = clf.predict_proba(X_test)[:, 1]

print("min %.4f  median %.4f  max %.4f"
      % (proba.min(), np.median(proba), proba.max()))
print("share above 0.50:", round((proba >= 0.50).mean(), 4))

min 0.0026  median 0.0423  max 0.6637
share above 0.50: 0.002


## 9.4 Precision, Recall, and the Cost of an Error

Accuracy hides which errors the model is making. Precision and recall separate them, and the costs of the collections process decide where the threshold goes.

### Code Example 9.10 — Precision and recall at the default threshold

In [11]:
from sklearn.metrics import precision_score, recall_score

print("precision:", round(precision_score(y_test, pred), 4))
print("recall:   ", round(recall_score(y_test, pred), 4))

precision: 0.3333
recall:    0.0106


### Code Example 9.11 — Precision and recall across candidate thresholds

In [12]:
rows = []
for t in [0.50, 0.30, 0.20, 0.15, 0.10, 0.06, 0.04]:
    flags = (proba >= t).astype(int)
    rows.append({
        "threshold": t,
        "flagged": int(flags.sum()),
        "precision": round(precision_score(y_test, flags, zero_division=0), 3),
        "recall": round(recall_score(y_test, flags), 3),
        "accuracy": round(accuracy_score(y_test, flags), 3),
    })
print(pd.DataFrame(rows).to_string(index=False))

 threshold  flagged  precision  recall  accuracy
      0.50        6      0.333   0.011     0.936
      0.30       46      0.391   0.095     0.934
      0.20      132      0.265   0.185     0.916
      0.15      260      0.238   0.328     0.892
      0.10      515      0.179   0.487     0.827
      0.06     1004      0.126   0.672     0.687
      0.04     1563      0.101   0.836     0.521


### Code Example 9.12 — The quarterly cost of a flagging rule

In [13]:
CASE_COST = 120.0
LOSS_RATE = 0.35
CURE_RATE = 0.55

amount = inv.loc[X_test.index, "invoice_amount"].to_numpy()
actual = y_test.to_numpy()

def quarter_cost(flags):
    cases = flags.sum() * CASE_COST
    missed = amount[(flags == 0) & (actual == 1)].sum() * LOSS_RATE
    worked = amount[(flags == 1) & (actual == 1)].sum() * LOSS_RATE * (1 - CURE_RATE)
    return cases + missed + worked

print("chase no one:    %10.2f" % quarter_cost(np.zeros(len(actual), dtype=int)))
print("chase everyone:  %10.2f" % quarter_cost(np.ones(len(actual), dtype=int)))
for t in [0.50, 0.30, 0.20, 0.15, 0.12, 0.10, 0.06]:
    print("threshold %.2f: %10.2f" % (t, quarter_cost((proba >= t).astype(int))))

chase no one:     677500.04
chase everyone:   664875.02
threshold 0.50:  674043.09
threshold 0.30:  636698.18
threshold 0.20:  588768.78
threshold 0.15:  522714.42
threshold 0.12:  489291.89
threshold 0.10:  487212.17
threshold 0.06:  502772.51


### Code Example 9.13 — Searching the threshold grid for the cheapest rule

In [14]:
grid = np.arange(0.01, 0.60, 0.005)
costs = np.array([quarter_cost((proba >= t).astype(int)) for t in grid])

print("best threshold: %.3f" % grid[costs.argmin()])
print("cost at best:   %.2f" % costs.min())
print("saving vs chase-no-one: %.2f" % (677500.04 - costs.min()))
print("saving share: %.4f" % (1 - costs.min() / 677500.04))

best threshold: 0.115
cost at best:   480967.65
saving vs chase-no-one: 196532.39
saving share: 0.2901


### Code Example 9.14 — The confusion matrix of the rule Priya will run

In [15]:
flags = (proba >= 0.12).astype(int)

print(confusion_matrix(y_test, flags))
print("precision:", round(precision_score(y_test, flags), 4))
print("recall:   ", round(recall_score(y_test, flags), 4))
print("accuracy: ", round(accuracy_score(y_test, flags), 4))

[[2502  309]
 [ 107   82]]
precision: 0.2097
recall:    0.4339
accuracy:  0.8613


### Code Example 9.15 — The area under the ROC curve

In [16]:
from sklearn.metrics import roc_auc_score

print("AUC:", round(roc_auc_score(y_test, proba), 4))

AUC: 0.7557


## 9.5 Trees, Margins, and Nonlinear Boundaries

Three methods that look for structure the logistic model assumes away, and what they find in this data.

### Code Example 9.16 — A depth-limited decision tree

In [17]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

tree = Pipeline([
    ("prep", prep),
    ("model", DecisionTreeClassifier(max_depth=4, random_state=0)),
])
tree.fit(X_train, y_train)

p_tree = tree.predict_proba(X_test)[:, 1]
print("accuracy:", round(accuracy_score(y_test, tree.predict(X_test)), 4))
print("AUC:     ", round(roc_auc_score(y_test, p_tree), 4))

accuracy: 0.935
AUC:      0.6722


### Code Example 9.17 — The same tree grown without a depth limit

In [18]:
deep = Pipeline([
    ("prep", prep),
    ("model", DecisionTreeClassifier(random_state=0)),
])
deep.fit(X_train, y_train)

print("training accuracy:", round(deep.score(X_train, y_train), 4))
print("test accuracy:    ", round(deep.score(X_test, y_test), 4))
print("test AUC:         ",
      round(roc_auc_score(y_test, deep.predict_proba(X_test)[:, 1]), 4))
print("leaves:", deep.named_steps["model"].get_n_leaves())

training accuracy: 1.0
test accuracy:     0.879
test AUC:          0.5406
leaves: 847


### Code Example 9.18 — A random forest on Priya's invoices

In [19]:
from sklearn.ensemble import RandomForestClassifier

forest = Pipeline([
    ("prep", prep),
    ("model", RandomForestClassifier(n_estimators=500, min_samples_leaf=50,
                                     random_state=0, n_jobs=-1)),
])
forest.fit(X_train, y_train)

p_forest = forest.predict_proba(X_test)[:, 1]
print("training accuracy:", round(forest.score(X_train, y_train), 4))
print("test accuracy:    ", round(accuracy_score(y_test, forest.predict(X_test)), 4))
print("AUC:              ", round(roc_auc_score(y_test, p_forest), 4))

training accuracy: 0.9369
test accuracy:     0.937
AUC:               0.7253


### Code Example 9.19 — Which variables the forest relied on

In [20]:
names = forest.named_steps["prep"].get_feature_names_out()
importance = pd.Series(
    forest.named_steps["model"].feature_importances_,
    index=[n.split("__")[1] for n in names],
).sort_values(ascending=False)

print(importance.head(6).round(4))

prior_late_rate        0.4080
log_amount             0.1833
relationship_months    0.1129
disputed_line_items    0.0953
prior_invoices         0.0582
order_channel_Phone    0.0384
dtype: float64


### Code Example 9.20 — A support vector machine with a radial basis kernel

In [21]:
from sklearn.svm import SVC

svm = Pipeline([
    ("prep", prep),
    ("model", SVC(probability=True, random_state=0)),
])
svm.fit(X_train, y_train)

print("accuracy:", round(accuracy_score(y_test, svm.predict(X_test)), 4))
print("flagged: ", int(svm.predict(X_test).sum()))
print("AUC:     ", round(roc_auc_score(y_test, svm.predict_proba(X_test)[:, 1]), 4))

accuracy: 0.937
flagged:  0
AUC:      0.6175


## 9.6 Validation, Leakage, and Honest Estimates

One split is one draw, and one column of the extract has been waiting since 9.1.

### Code Example 9.21 — Five-fold stratified cross-validation

In [22]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(clf, X, y, cv=cv, scoring="roc_auc")

print(np.round(scores, 4))
print("mean %.4f  sd %.4f" % (scores.mean(), scores.std()))

[0.7477 0.7509 0.7152 0.7785 0.7569]
mean 0.7499  sd 0.0204


### Code Example 9.22 — The model with the contact column added

In [23]:
leak_num = NUM + ["collections_contact_day"]
inv_leak = inv.copy()
inv_leak["collections_contact_day"] = inv_leak["collections_contact_day"].fillna(-1)

XL = inv_leak[leak_num + CAT]
XL_train, XL_test, yL_train, yL_test = train_test_split(
    XL, y, test_size=0.25, stratify=y, random_state=42)

prep_leak = ColumnTransformer([
    ("num", StandardScaler(), leak_num),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), CAT),
])
leaky = Pipeline([("prep", prep_leak), ("model", LogisticRegression(max_iter=3000))])
leaky.fit(XL_train, yL_train)

print("accuracy:", round(accuracy_score(yL_test, leaky.predict(XL_test)), 4))
print("recall:  ", round(recall_score(yL_test, leaky.predict(XL_test)), 4))
print("AUC:     ", round(roc_auc_score(yL_test, leaky.predict_proba(XL_test)[:, 1]), 4))

accuracy: 0.9823
recall:   0.8519
AUC:      0.9853


### Code Example 9.23 — Where the contact column comes from

In [24]:
print(inv.groupby("delinquent_90")["collections_contact_day"]
        .agg(non_null="count", share=lambda s: round(s.notna().mean(), 4)))

               non_null  share
delinquent_90                 
0                   135  0.012
1                   704  0.930


## Where this leaves Priya

The operating rule flags invoices whose estimated probability exceeds 0.12. It opens about 391 cases a quarter, catches 43% of the delinquencies, and saves roughly \$196,000 against leaving the book alone — at an accuracy seven points *below* the rule that flags nothing.

Try the exercises at the end of the chapter in this notebook: the cost parameters, the capacity-constrained rule, and the forest's leaf size are all one edit away from the cells above.